In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text, inspect
from sqlalchemy import Integer, String, Float, Boolean, DateTime
import urllib.parse
import logging
import glob
import os
import sys


# ________ Logging Setup __________________________________________________________________

def setup_logging(log_file="UPI_Transactions_Pipeline.log"):
    logger = logging.getLogger("UPITransactionPipeline")

    if logger.handlers:          # prevents duplicate handlers on re-run in Jupyter
        return logger

    logger.setLevel(logging.DEBUG)
    logger.propagate = False     # stops bubbling to Jupyter root logger (eliminates duplicate lines)

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(log_file, mode='a', encoding='utf-8')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    return logger


# ______ Stage 1 : Ingestion ________________________________________________________________

def ingest(folder_path, logger):
    logger.info("Stage 1 | Ingestion Started")
    logger.debug(f"Reading CSVs From : {folder_path}")

    if not os.path.exists(folder_path):
        logger.error(f"Folder not Found : {folder_path}")
        raise FileNotFoundError(f"Folder not Found : {folder_path}")

    dataframes = {}
    files = glob.glob(os.path.join(folder_path, "*.csv"))

    if not files:
        logger.warning("No CSV file found in folder")
        return dataframes

    for file in files:
        name = os.path.basename(file).replace(".csv", "")
        try:
            dataframes[name] = pd.read_csv(file)
            logger.debug(f"Loaded '{name}' -> {dataframes[name].shape[0]} rows , {dataframes[name].shape[1]} cols")
        except Exception as e:
            logger.error(f"Failed to read '{name}': {e}")
            raise

    logger.info(f"Stage 1 | Ingestion Completed -> {len(dataframes)} files loaded : {list(dataframes.keys())}")
    return dataframes


# _______ Stage 2 : Clean customer_details ____________________________________________________

def Clean_customer_details(df, logger):
    logger.info("Stage 2 | Cleaning Customer Details")
    df = df.copy()

    before = df.shape[0]
    df['date_joined'] = pd.to_datetime(df['date_joined'])
    logger.debug("Converted date_joined to datetime")

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null Values after cleaning : {nulls}")
    logger.info(f"customer_details -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# _______ Stage 3 : Clean customer_feedback ____________________________________________________

def Clean_customer_feedback(df, logger):
    logger.info("Stage 3 | Cleaning customer_feedback")
    df = df.copy()

    before = df.shape[0]
    df['date_submitted'] = pd.to_datetime(df['date_submitted'])
    logger.debug("Converted date_submitted to datetime")

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"customer_feedback -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# ________ Stage 4 : Clean device_info _____________________________________________________________

def Clean_device_info(df, logger):
    logger.info("Stage 4 | Cleaning device_info")
    df = df.copy()

    before = df.shape[0]
    df['last_active'] = pd.to_datetime(df['last_active'])
    logger.debug("Converted last_active to datetime")

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"device_info -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# _________ Stage 5 : Clean fraud_alert _______________________________________________________________

def Clean_fraud_alert(df, logger):
    logger.info("Stage 5 | Cleaning fraud_alert")
    df = df.copy()

    before = df.shape[0]
    df['alert_date']      = pd.to_datetime(df['alert_date'])
    # FIX 4 : resolution_date was never being converted to datetime — added here
    df['resolution_date'] = pd.to_datetime(df['resolution_date'], errors='coerce')
    logger.debug("Converted alert_date and resolution_date to datetime")

    # NOTE : resolution_date nulls are expected — unresolved alerts (resolved=False). NOT missing data.

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"fraud_alert -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# __________ Stage 6 : Clean merchant __________________________________________________________________

def Clean_merchant(df, logger):
    logger.info("Stage 6 | Cleaning merchant")
    df = df.copy()

    before = df.shape[0]
    df['onboard_date'] = pd.to_datetime(df['onboard_date'])
    logger.debug("Converted onboard_date to datetime")

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"merchant -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# __________ Stage 7 : Clean account_details _____________________________________________________________

def Clean_account_details(df, logger):
    logger.info("Stage 7 | Cleaning account_details")
    df = df.copy()

    before = df.shape[0]
    df['date_added'] = pd.to_datetime(df['date_added'])
    logger.debug("Converted date_added to datetime")

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"account_details -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# ____________ Stage 8 : Clean transaction_history ________________________________________________________

def Clean_transaction_history(df, logger):
    logger.info("Stage 8 | Cleaning transaction_history")
    df = df.copy()

    before = df.shape[0]
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    logger.debug("Converted timestamp to datetime")

    # NOTE : merchant_id nulls are expected — send/receive txns have no merchant
    # NOTE : failure_reason nulls are expected — only failed txns have a reason

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"transaction_history -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# __ Stage 9 : Database Engine _______________________________________________________________
# FIX 5 : Removed fast_executemany=True — it causes datetime microsecond truncation bug
#          with SQL Server, causing 'String data, right truncation' error on fraud_alert

def create_db_engine(server_name, database_name, logger):
    logger.info("Stage 9 | Creating database engine...")
    try:
        params = urllib.parse.quote_plus(
            f'DRIVER={{ODBC Driver 18 for SQL Server}};'
            f'SERVER={server_name};'
            f'DATABASE={database_name};'
            f'Trusted_Connection=yes;'
            f'Encrypt=yes;'
            f'TrustServerCertificate=yes;'
        )
        engine = create_engine(
            f"mssql+pyodbc:///?odbc_connect={params}"
            # fast_executemany removed — causes datetime truncation with microseconds
        )
        # Test connection
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        logger.info("  Database engine created and connection verified.")
        return engine
    except Exception as e:
        logger.error(f"  Failed to create engine: {e}")
        raise


# __ Stage 10 : Smart Upload _________________________________________________________________
# FIX 6 : upload uses engine.begin() for proper transaction rollback on failure
# FIX 7 : "Smart upload complete - -" typo fixed (double dash)
# FIX 8 : Row count validation uses [0] index safely

def upload_to_sql(df, table_name, engine, unique_keys, dtype_map, logger):
    logger.info(f"  Uploading '{table_name}' - {len(df)} records...")

    try:
        inspector    = inspect(engine)
        table_exists = inspector.has_table(table_name)

        # CASE 1 - Fresh upload (table does not exist)
        if not table_exists:
            logger.info(f"  Table '{table_name}' not found - fresh upload...")
            df["uploaded_at"] = pd.Timestamp.now()
            with engine.begin() as conn:
                df.to_sql(
                    table_name,
                    conn,
                    if_exists="replace",
                    index=False,
                    chunksize=10000,
                    dtype=dtype_map
                )
            logger.info(f"  Fresh upload complete - {len(df)} records inserted.")

        # CASE 2 - Smart insert (table exists, skip duplicates)
        else:
            logger.info(f"  Table '{table_name}' exists - checking for duplicates...")

            existing_df = pd.read_sql(
                f"SELECT {', '.join(unique_keys)} FROM {table_name}",
                engine
            )
            logger.info(f"  Existing records in DB: {len(existing_df)}")

            merged   = df.merge(existing_df, on=unique_keys, how="left", indicator=True)
            new_rows = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

            if len(new_rows) == 0:
                logger.info(f"  No new records for '{table_name}' - upload skipped.")
                return

            new_rows["uploaded_at"] = pd.Timestamp.now()
            with engine.begin() as conn:
                new_rows.to_sql(
                    table_name,
                    conn,
                    if_exists="append",
                    index=False,
                    chunksize=10000,
                    dtype=dtype_map
                )
            logger.info(
                f"  Smart upload complete - "
                f"{len(new_rows)} new records inserted, "
                f"{len(df) - len(new_rows)} duplicates skipped."
            )

        # Row Count Validation
        db_count = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {table_name}", engine).iloc[0]['cnt']
        logger.info(f"  [OK] Row count validation - DB now has {db_count} records in '{table_name}'.")

    except Exception as e:
        logger.error(f"  Upload failed for '{table_name}': {e}")
        raise


# _________ dtype_maps _____________________________________________________________________________

DTYPE_MAPS = {
    "customer_details": {
        "customer_id":      String(50),
        "full_name":        String(200),
        "mobile_number":    String(20),
        "age":              Integer(),
        "gender":           String(20),
        "region":           String(50),
        "date_joined":      DateTime(),
        "is_business_user": Boolean(),
        "risk_score":       Float()
    },
    "customer_feedback": {
        "feedback_id":        String(50),
        "customer_id":        String(50),
        "date_submitted":     DateTime(),
        "feedback_text":      String(500),
        "satisfaction_score": Integer(),
        "issue_type":         String(50),
        "resolved":           Boolean()
    },
    "device_info": {
        "device_id":   String(50),
        "customer_id": String(50),
        "device_type": String(50),
        "app_version": String(20),
        "is_rooted":   Boolean(),
        "last_active": DateTime()
    },
    "fraud_alert": {
        "alert_id":        String(50),
        "transaction_id":  String(50),
        "alert_type":      String(50),
        "alert_date":      DateTime(),
        "resolved":        Boolean(),
        "resolution_date": DateTime(),
        "remarks":         String(500)
    },
    "merchant": {
        "merchant_id":   String(50),
        "merchant_name": String(200),
        "merchant_type": String(50),
        "region":        String(50),
        "onboard_date":  DateTime(),
        "risk_score":    Float()
    },
    "account_details": {
        "upi_id":       String(50),
        "customer_id":  String(50),
        "bank_name":    String(50),
        "account_type": String(50),
        "date_added":   DateTime(),
        "status":       String(20)
    },
    "transaction_history": {
        "transaction_id":   String(50),
        "upi_id":           String(50),
        "customer_id":      String(50),
        "timestamp":        DateTime(),
        "amount":           Float(),
        "transaction_type": String(50),
        "merchant_id":      String(50),
        "counterparty_upi": String(50),
        "status":           String(20),
        "device_id":        String(50),
        "device_type":      String(50),
        "channel":          String(50),
        "fraud_flag":       Boolean(),
        "reversal_flag":    Boolean(),
        "failure_reason":   String(100)
    }
}


# ________ Unique Keys ________________________________________________________________________

UNIQUE_KEYS = {
    "customer_details":    ["customer_id"],
    "customer_feedback":   ["feedback_id"],
    "device_info":         ["device_id"],
    "fraud_alert":         ["alert_id"],
    "merchant":            ["merchant_id"],
    "account_details":     ["upi_id"],
    "transaction_history": ["transaction_id"]
}


# ________ Master Pipeline _____________________________________________________________________

def run_pipeline(folder_path, server_name, database_name, log_file="UPI_Transactions_Pipeline.log"):
    logger = setup_logging(log_file)
    logger.info('=' * 70)
    logger.info("UPI Transactions Pipeline - START")
    logger.info('=' * 70)

    try:
        # Stage 1 - Ingestion
        raw = ingest(folder_path, logger)

        # Stage 2-8 - Clean all tables
        customer_details    = Clean_customer_details(raw['customer_master'], logger)
        customer_feedback   = Clean_customer_feedback(raw['customer_feedback_surveys'], logger)
        device_info         = Clean_device_info(raw['device_info'], logger)
        fraud_alert         = Clean_fraud_alert(raw['fraud_alert_history'], logger)
        merchant            = Clean_merchant(raw['merchant_info'], logger)
        account_details     = Clean_account_details(raw['upi_account_details'], logger)
        transaction_history = Clean_transaction_history(raw['upi_transaction_history'], logger)

        # Stage 9 - Database engine
        engine = create_db_engine(server_name, database_name, logger)

        # Stage 10 - Upload all tables
        logger.info("Stage 10 | Uploading all tables to SQL Server")
        upload_to_sql(customer_details,     "customer_details",     engine, UNIQUE_KEYS["customer_details"],     DTYPE_MAPS["customer_details"],     logger)
        upload_to_sql(customer_feedback,    "customer_feedback",    engine, UNIQUE_KEYS["customer_feedback"],    DTYPE_MAPS["customer_feedback"],    logger)
        upload_to_sql(device_info,          "device_info",          engine, UNIQUE_KEYS["device_info"],          DTYPE_MAPS["device_info"],          logger)
        upload_to_sql(fraud_alert,          "fraud_alert",          engine, UNIQUE_KEYS["fraud_alert"],          DTYPE_MAPS["fraud_alert"],          logger)
        upload_to_sql(merchant,             "merchant",             engine, UNIQUE_KEYS["merchant"],             DTYPE_MAPS["merchant"],             logger)
        upload_to_sql(account_details,      "account_details",      engine, UNIQUE_KEYS["account_details"],      DTYPE_MAPS["account_details"],      logger)
        upload_to_sql(transaction_history,  "transaction_history",  engine, UNIQUE_KEYS["transaction_history"],  DTYPE_MAPS["transaction_history"],  logger)

        logger.info('=' * 70)
        logger.info("UPI Transactions Pipeline - COMPLETE")
        logger.info('=' * 70)

    except Exception as e:
        logger.critical(f"Pipeline FAILED : {e}", exc_info=True)
        raise


In [2]:
# _________ Run ________________________________________________________________________________________
# NOTE : Before first run, drop any partially-created tables from failed previous runs:
#
#   from sqlalchemy import create_engine, text
#   import urllib.parse
#   params = urllib.parse.quote_plus(
#       'DRIVER={ODBC Driver 18 for SQL Server};SERVER=LAPTOP-2VUPGIQA\\SQLEXPRESS;'
#       'DATABASE=UPI_Transactions_Data_DB;Trusted_Connection=yes;Encrypt=yes;TrustServerCertificate=yes;'
#   )
#   engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
#   with engine.begin() as conn:
#       for t in ['customer_details','customer_feedback','device_info',
#                 'fraud_alert','merchant','account_details','transaction_history']:
#           conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
#   print("All tables dropped - ready for fresh run")

folder_path   = r'D:\DATA SETS (Project)\Capston Project Data\UPI_Transactions_Data_Analysis'
server_name   = r'LAPTOP-2VUPGIQA\SQLEXPRESS'
database_name = 'UPI_Transactions_Data_DB'

run_pipeline(folder_path, server_name, database_name, log_file="UPI_Transactions_Pipeline.log")


2026-05-30 03:46:50 | INFO     | UPITransactionPipeline | ======================================================================
2026-05-30 03:46:50 | INFO     | UPITransactionPipeline | UPI Transactions Pipeline - START
2026-05-30 03:46:50 | INFO     | UPITransactionPipeline | ======================================================================
2026-05-30 03:46:50 | INFO     | UPITransactionPipeline | Stage 1 | Ingestion Started
2026-05-30 03:46:51 | INFO     | UPITransactionPipeline | Stage 1 | Ingestion Completed -> 7 files loaded : ['customer_feedback_surveys', 'customer_master', 'device_info', 'fraud_alert_history', 'merchant_info', 'upi_account_details', 'upi_transaction_history']
2026-05-30 03:46:51 | INFO     | UPITransactionPipeline | Stage 2 | Cleaning Customer Details
2026-05-30 03:46:51 | INFO     | UPITransactionPipeline | customer_details -> 10000 rows ( no rows dropped from 10000 )
2026-05-30 03:46:51 | INFO     | UPITransactionPipeline | Stage 3 | Cleaning customer_fe

C:\Users\user\AppData\Local\Temp\ipykernel_27560\4078681682.py:212: SAWarning: Unrecognized server version info '17.0.1115.1'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


2026-05-30 03:46:52 | INFO     | UPITransactionPipeline |   Table 'customer_details' exists - checking for duplicates...
2026-05-30 03:46:52 | INFO     | UPITransactionPipeline |   Existing records in DB: 10000
2026-05-30 03:46:52 | INFO     | UPITransactionPipeline |   No new records for 'customer_details' - upload skipped.
2026-05-30 03:46:52 | INFO     | UPITransactionPipeline |   Uploading 'customer_feedback' - 4000 records...
2026-05-30 03:46:52 | INFO     | UPITransactionPipeline |   Table 'customer_feedback' exists - checking for duplicates...
2026-05-30 03:46:52 | INFO     | UPITransactionPipeline |   Existing records in DB: 4000
2026-05-30 03:46:52 | INFO     | UPITransactionPipeline |   No new records for 'customer_feedback' - upload skipped.
2026-05-30 03:46:52 | INFO     | UPITransactionPipeline |   Uploading 'device_info' - 12000 records...
2026-05-30 03:46:52 | INFO     | UPITransactionPipeline |   Table 'device_info' exists - checking for duplicates...
2026-05-30 03:46:5